In [ ]:
# 1. Introduction
%pip install python-crfsuite openpyxl

In [ ]:
# 2. Importing Libraries
import pycrfsuite
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
# 3. Loading Data
df = pd.read_excel('./crf.xlsx')
df['token'] = df['token'].astype(str)
df['classification'] = df['classification'].astype(str)
print(f'Shape: {df.shape[0]}')

Shape: 4153


In [14]:
# 4. Function Definitions

def transform_df_to_training_data(dataframe=df):
  results = []
  actual_string = []
  last_idx = None
  for _, row in dataframe.iterrows():
    if last_idx is None:
      last_idx = row['idx']
    if len(actual_string) == 0 or row['idx'] == last_idx:
      actual_string.append((row['token'], row['classification']))
    else:
      results.append(actual_string)
      actual_string = [(row['token'], row['classification'])]
      last_idx = row['idx']
  if actual_string:
    results.append(actual_string)
  return results

def validate_cpf(cpf: str) -> bool:
  cpf = cpf.replace(".", "").replace("-", "")
  if len(cpf) != 11 or not cpf.isdigit():
    return False
  if cpf == cpf[0] * 11:
    return False
  def calculate_digit(partial, weight):
    total = sum(int(d)*w for d, w in zip(partial, range(weight, 1, -1)))
    r = total % 11
    return '0' if r < 2 else str(11 - r)
  d1 = calculate_digit(cpf[:9], 10)
  d2 = calculate_digit(cpf[:9] + d1, 11)
  return cpf[-2:] == d1 + d2

def validate_cnpj(cnpj: str) -> bool:
  cnpj = cnpj.replace(".", "").replace("-", "").replace("/", "")
  if len(cnpj) != 14 or not cnpj.isdigit():
    return False
  if cnpj == cnpj[0] * 14:
    return False
  def calc(partial, weights):
    total = sum(int(d)*w for d, w in zip(partial, weights))
    r = total % 11
    return '0' if r < 2 else str(11 - r)
  w1 = [5,4,3,2,9,8,7,6,5,4,3,2]
  w2 = [6,5,4,3,2,9,8,7,6,5,4,3,2]
  d1 = calc(cnpj[:12], w1)
  d2 = calc(cnpj[:12] + d1, w2)
  return cnpj[-2:] == d1 + d2

def word2features(sent, i):
  word = sent[i][0]
  feats = {
    'bias': 1.0,
    'word.lower()': word.lower(),
    'word.isupper()': word.isupper(),
    'word.istitle()': word.istitle(),
    'word.isdigit()': word.isdigit(),
    'prefix-1': word[0],
    'suffix-1': word[-1],
    'prefix-2': word[:2],
    'suffix-2': word[-2:],
    'prefix-3': word[:3],
    'suffix-3': word[-3:],
    'is_cpf': validate_cpf(word),
    'is_cnpj': validate_cnpj(word)
  }
  if i > 0:
    w1 = sent[i-1][0]
    feats.update({
      '-1:word.lower()': w1.lower(),
      '-1:word.istitle()': w1.istitle(),
      '-1:word.isupper()': w1.isupper()
    })
  else:
    feats['BOS'] = True
  if i < len(sent)-1:
    w2 = sent[i+1][0]
    feats.update({
      '+1:word.lower()': w2.lower(),
      '+1:word.istitle()': w2.istitle(),
      '+1:word.isupper()': w2.isupper()
    })
  else:
    feats['EOS'] = True
  return feats

def sent2features(sent):
  return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
  return [label for _, label in sent]

def sent2tokens(sent):
  return [token for token, _ in sent]

In [ ]:
# 5. Training and Saving Model

# Transform data
training = transform_df_to_training_data(df)
X = [sent2features(s) for s in training]
y = [sent2labels(s) for s in training]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train CRF
trainer = pycrfsuite.Trainer(verbose=True)
for xseq, yseq in zip(X_train, y_train):
  trainer.append(xseq, yseq)

trainer.set_params({
  'c1': 0.1,
  'c2': 0.1,
  'max_iterations': 100,
  'feature.possible_transitions': True
})

model_path = 'ner-crf.model'
trainer.train(model_path)
print(f'Model saved to {model_path}')

# Evaluate
tagger = pycrfsuite.Tagger()
tagger.open(model_path)
y_pred = [tagger.tag(xseq) for xseq in X_test]

flat_test = [lbl for seq in y_test for lbl in seq]
flat_pred = [lbl for seq in y_pred for lbl in seq]
accuracy = accuracy_score(flat_test, flat_pred)
report = classification_report(flat_test, flat_pred, output_dict=True)

print(f'Accuracy: {accuracy:.4f}')
print(classification_report(flat_test, flat_pred))                                                                                                                                                                                                                                                          

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 10999
Seconds required: 0.009

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 100
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 7436.838889
Feature norm: 1.000000
Error norm: 2355.733346
Active features: 10952
Line search trials: 1
Line search step: 0.000360
Seconds required for this iteration: 0.006

***** Iteration #2 *****
Loss: 5097.737580
Feature norm: 4.267968
Error norm: 1785.262836
Active features: 10900
Line search trials: 2
Line search step: 0.500000
Seconds required for this iteration: 0.007

***** Iteration #3 *****
Loss: 3974.786396
Feature norm: 4.122177
Error norm: 552.304764
Active features: 10629
Line search trials: 1
Line search step: 1.000000
Seconds required for this i

/home/gustavo/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/gustavo/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/gustavo/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/ho

In [19]:
# 6. Utility for Prediction

def predict_entities(text: str):
  tokens = text.split()
  features = sent2features([(t, '') for t in tokens])
  tagger_local = pycrfsuite.Tagger()
  tagger_local.open(model_path)
  labels = tagger_local.tag(features)
  return [{"token": t, "classification": l} for t, l in zip(tokens, labels)]

In [20]:
# 7. Testing Model Locally

if __name__ == '__main__':
  sample = "Rua Irineu Ferreira da Silva, 231 Taubaté São Paulo CEP"
  result = predict_entities(sample)
  print(result)


[{'token': 'Rua', 'classification': 'LX'}, {'token': 'Irineu', 'classification': 'street_name'}, {'token': 'Ferreira', 'classification': 'LX'}, {'token': 'da', 'classification': 'street_type'}, {'token': 'Silva,', 'classification': 'LX'}, {'token': '231', 'classification': 'address_number'}, {'token': 'Taubaté', 'classification': 'city'}, {'token': 'São', 'classification': 'suburb'}, {'token': 'Paulo', 'classification': 'LX'}, {'token': 'CEP', 'classification': 'antecessor_zip'}]
